# Linear Regression (CFA Level 1)
## OLS Derivation, Diagnostics, and Financial Applications

This notebook provides a rigorous, from-scratch treatment of **linear regression** as covered in the CFA Level 1 curriculum, with applications to the Capital Asset Pricing Model (CAPM) and factor models.

**What you will learn:**
1. Simple linear regression: OLS derivation via calculus
2. Regression diagnostics: $R^2$, standard error, residual analysis
3. Hypothesis testing for coefficients (t-tests, F-tests)
4. ANOVA decomposition
5. Assumption violations and detection methods
6. Multiple regression via matrix algebra
7. CAPM beta estimation with simulated market data

**Why it matters:** Regression is the workhorse of quantitative finance -- used to estimate risk exposures (betas), evaluate factor models, and forecast returns.

**Prerequisites:** Basic calculus (partial derivatives), matrix algebra.

**References:**
- CFA Institute, *CFA Program Curriculum*, Quantitative Methods.
- DeFusco, R. et al., *Quantitative Investment Analysis*, CFA Institute, Wiley.

---
## Why Regression Matters in Finance

Linear regression is the single most important statistical technique in empirical finance. Almost every major model — CAPM, multi-factor models, event studies, yield curve analysis — boils down to regression at some level. If you truly understand regression, you have the key that unlocks most of quantitative finance.

### What Does Regression Actually Do?

Imagine you have a scatter plot of two variables — say, the monthly return on Apple stock (Y-axis) and the monthly return on the S&P 500 (X-axis). Each dot represents one month. The dots form a cloud. **Regression draws the single best-fitting straight line through that cloud.**

> **Key Concept:** At its core, regression answers the question: **"What is the best-fitting straight line through this data?"** The "best" line is the one that minimises the sum of squared vertical distances from the data points to the line.

But why do we care about a straight line?

- **Summarisation:** The line summarises the average relationship between X and Y in just two numbers: the slope and the intercept.
- **Prediction:** Given a new value of X (say, the market returned +3% this month), we can predict Y (what Apple's return might be).
- **Inference:** We can ask whether the relationship is "real" (statistically significant) or could have arisen by chance.

### A Concrete Financial Example

Suppose you regress Apple's monthly excess returns on the S&P 500's monthly excess returns and find:

$$R_{\text{Apple}} - R_f = 0.005 + 1.2 \times (R_{\text{S\&P}} - R_f)$$

This single equation tells you:

| Parameter | Value | Financial Meaning |
|:----------|:------|:------------------|
| Intercept (alpha) | 0.005 | Apple earns 0.5% per month *above* what CAPM predicts — positive abnormal return |
| Slope (beta) | 1.2 | Apple is 20% more volatile than the market — when the market goes up 1%, Apple tends to go up 1.2% |

That is the power of regression: two numbers that capture the risk profile of a stock.

> **CFA Exam Tip:** On the exam, you will be asked to *interpret* regression output — not just compute it. Practice translating numbers into plain English. "A beta of 1.2 means that for every 1% increase in the market excess return, the stock's excess return increases by 1.2%, on average."

---
### The OLS Framework

The simple linear regression model is:

$$Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i$$

where:
- $Y_i$ = dependent variable (what we are trying to explain — e.g., stock return)
- $X_i$ = independent variable (what we are using to explain it — e.g., market return)
- $\beta_0$ = intercept (value of Y when X = 0)
- $\beta_1$ = slope (change in Y per one-unit change in X)
- $\varepsilon_i$ = error term (everything else that affects Y but is not captured by X)

Think of the error term as the "catchall" — all the reasons Apple's return might differ from what the market alone would predict (company-specific news, earnings surprises, analyst upgrades, etc.).

### Why "Ordinary Least Squares"?

**OLS (Ordinary Least Squares)** finds the $\beta_0$ and $\beta_1$ that minimise:

$$\sum_{i=1}^n (Y_i - \hat{Y}_i)^2 = \sum_{i=1}^n (Y_i - \hat{\beta}_0 - \hat{\beta}_1 X_i)^2$$

This raises two natural questions:

**Why squared residuals?** Why not just minimise the *absolute* distances $|Y_i - \hat{Y}_i|$?

1. **Mathematical convenience:** Squaring gives a smooth, differentiable function that we can optimise with calculus. Absolute values create a "kink" at zero that is harder to work with.
2. **Penalises large errors more:** A residual of 10 contributes $10^2 = 100$ to the sum of squares, but a residual of 1 contributes only $1^2 = 1$. So OLS is especially keen to avoid large outliers.
3. **Connection to variance:** The sum of squared residuals is directly related to the variance of the errors, linking OLS to the theory of maximum likelihood estimation under normality.

> **Common Mistake:** Students sometimes wonder, "why not minimise *horizontal* distances?" Because in regression, X and Y play asymmetric roles. We are explaining Y *using* X, so we measure how far each observed Y is from its predicted value — which is a *vertical* distance on the scatter plot.

**Why vertical distances?**

Consider a scatter plot. The regression line predicts $\hat{Y}$ for each $X$. The residual $e_i = Y_i - \hat{Y}_i$ is the vertical gap between the observed point and the line. We minimise the sum of these squared vertical gaps because:

- We treat X as "given" (the independent variable) and Y as "random" (the dependent variable).
- We want to predict Y as accurately as possible, conditional on knowing X.
- If we minimised horizontal distances, we would be predicting X from Y — a different regression entirely.

### Deriving the OLS Formulas

Taking partial derivatives and setting them to zero:

$$\frac{\partial \text{SSE}}{\partial \beta_0} = -2\sum_{i=1}^n (Y_i - \beta_0 - \beta_1 X_i) = 0$$

$$\frac{\partial \text{SSE}}{\partial \beta_1} = -2\sum_{i=1}^n X_i(Y_i - \beta_0 - \beta_1 X_i) = 0$$

From the first equation (the "normal equation" for the intercept):

$$\sum Y_i = n\hat{\beta}_0 + \hat{\beta}_1 \sum X_i$$
$$\bar{Y} = \hat{\beta}_0 + \hat{\beta}_1 \bar{X}$$
$$\hat{\beta}_0 = \bar{Y} - \hat{\beta}_1 \bar{X}$$

This tells us something beautiful: **the regression line always passes through the point $(\bar{X}, \bar{Y})$** — the "centre of gravity" of the data.

Substituting $\hat{\beta}_0$ into the second equation and solving:

$$\hat{\beta}_1 = \frac{\sum(X_i - \bar{X})(Y_i - \bar{Y})}{\sum(X_i - \bar{X})^2} = \frac{\text{Cov}(X, Y)}{\text{Var}(X)}$$

> **Key Concept:** The slope is the covariance of X and Y divided by the variance of X. In CAPM terms, $\beta = \text{Cov}(R_i, R_m) / \text{Var}(R_m)$ — exactly the same formula! This is not a coincidence. CAPM beta IS the regression slope.

### Worked Example: Slope Calculation by Hand

Suppose we have 5 months of data:

| Month | Market Return (X) | Stock Return (Y) |
|:------|:----------------:|:----------------:|
| 1 | 2% | 3% |
| 2 | -1% | -2% |
| 3 | 4% | 6% |
| 4 | 0% | 1% |
| 5 | 3% | 4% |

**Step 1:** Compute means: $\bar{X} = (2 - 1 + 4 + 0 + 3)/5 = 1.6\%$, $\bar{Y} = (3 - 2 + 6 + 1 + 4)/5 = 2.4\%$

**Step 2:** Compute deviations and cross-products:

| Month | $X_i - \bar{X}$ | $Y_i - \bar{Y}$ | Product | $(X_i - \bar{X})^2$ |
|:------|:-----:|:-----:|:-----:|:-----:|
| 1 | 0.4 | 0.6 | 0.24 | 0.16 |
| 2 | -2.6 | -4.4 | 11.44 | 6.76 |
| 3 | 2.4 | 3.6 | 8.64 | 5.76 |
| 4 | -1.6 | -1.4 | 2.24 | 2.56 |
| 5 | 1.4 | 1.6 | 2.24 | 1.96 |
| **Sum** | | | **24.80** | **17.20** |

**Step 3:** $\hat{\beta}_1 = 24.80 / 17.20 = 1.442$

**Step 4:** $\hat{\beta}_0 = 2.4 - 1.442 \times 1.6 = 0.093$

Interpretation: This stock has a beta of approximately 1.44 — it moves about 44% more than the market. The intercept of 0.093% suggests a small positive alpha.

> **CFA Exam Tip:** You may need to compute the slope by hand on the exam. The formula $\hat{\beta}_1 = \sum(X_i - \bar{X})(Y_i - \bar{Y}) / \sum(X_i - \bar{X})^2$ must be memorised. Remember: numerator is the cross-products, denominator is the squared deviations of X only.

---
### R-squared: How Good Is the Fit?

Once we have fitted the line, the obvious question is: **how well does it fit?** This is where $R^2$ comes in.

$$R^2 = 1 - \frac{\text{SSE}}{\text{SST}} = 1 - \frac{\sum(Y_i - \hat{Y}_i)^2}{\sum(Y_i - \bar{Y})^2}$$

#### What Does R-squared Really Mean?

> **Key Concept:** $R^2$ is the **percentage of variation in Y that is explained by X**. If $R^2 = 0.75$, the regression explains 75% of the variation in Y; the remaining 25% is unexplained (due to other factors or randomness).

Think of it this way:

- **Before regression:** The best you can do to predict Y is the mean $\bar{Y}$. The total variation around this mean is SST.
- **After regression:** You predict $\hat{Y}_i = \hat{\beta}_0 + \hat{\beta}_1 X_i$. The remaining variation around these predictions is SSE.
- **$R^2$** measures how much of the original variation you have "explained away" by using X.

#### Worked Example: R-squared with a Tiny Dataset

Using our 5-month example above, suppose we compute:

- $\text{SST} = \sum(Y_i - \bar{Y})^2 = 0.36 + 19.36 + 12.96 + 1.96 + 2.56 = 37.20$
- $\text{SSE} = \sum(Y_i - \hat{Y}_i)^2 = 0.78$ (after computing predicted values from our line)
- $R^2 = 1 - 0.78 / 37.20 = 0.979$

This means the market return explains about 97.9% of the variation in this stock's return — an extremely strong relationship (typical of a large-cap stock that closely tracks the index).

#### What Counts as a "Good" R-squared?

This depends entirely on context:

| Context | Typical R-squared | Why |
|:--------|:-------:|:----|
| CAPM regression (single stock vs market) | 0.20 - 0.60 | Lots of idiosyncratic risk |
| Portfolio vs benchmark | 0.85 - 0.99 | Portfolios diversify away idiosyncratic risk |
| Cross-sectional return prediction | 0.01 - 0.05 | Predicting returns is hard! |
| Bond yield vs maturity | 0.90+ | Strong structural relationship |

> **Common Mistake:** A high $R^2$ does NOT mean causation. Two variables can be highly correlated for many reasons (common factor, coincidence, reverse causation). Always think about the economic story behind the relationship. Also, never judge a model solely by $R^2$ — a model with a high $R^2$ can still have biased coefficients, violated assumptions, or be completely useless for out-of-sample prediction.

> **CFA Exam Tip:** In simple regression, $R^2$ equals the square of the correlation coefficient: $R^2 = r_{XY}^2$. If the correlation between X and Y is 0.8, then $R^2 = 0.64$. This relationship holds ONLY for simple regression (one independent variable), NOT for multiple regression.

---
### The ANOVA Decomposition

The total variation in Y can be decomposed into two components:

$$\underbrace{\text{SST}}_{\text{Total variation}} = \underbrace{\text{SSR}}_{\text{Explained by regression}} + \underbrace{\text{SSE}}_{\text{Unexplained (residual)}}$$

Think of ANOVA as an **accounting identity for variation**:

- **SST (Total Sum of Squares):** How much does Y vary overall? This is the variation you started with.
- **SSR (Regression Sum of Squares):** How much of that variation can the regression line account for? This is the "signal."
- **SSE (Error Sum of Squares):** How much variation remains after the regression? This is the "noise."

The F-test in the ANOVA table asks: **is the signal large enough relative to the noise that we can conclude the regression is meaningful?**

> **Key Concept:** ANOVA is about separating **signal** from **noise**. The F-statistic is the ratio of "explained variation per degree of freedom" to "unexplained variation per degree of freedom." A large F means the regression explains much more than random chance would suggest.

---
### Key Assumptions of OLS

For the standard inference (t-tests, confidence intervals) to be valid, we need four key assumptions. Understanding **why** each matters — and what goes wrong when it is violated — is critical for both the exam and real-world practice.

#### 1. Linearity: The Relationship Is a Straight Line

**What it means:** The expected value of Y, conditional on X, falls on a straight line: $E[Y|X] = \beta_0 + \beta_1 X$.

**Why it matters:** If the true relationship is curved (say, logarithmic or quadratic), forcing a straight line through the data produces **biased estimates**. The slope and intercept will be systematically wrong, not just noisy.

**What goes wrong:** Your predictions will be too high in some regions and too low in others. The residual plot will show a U-shape or inverted U-shape pattern instead of random scatter.

**Financial example:** The relationship between option price and underlying price is highly nonlinear (Black-Scholes is an exponential formula). Fitting a linear regression here would give misleading results.

#### 2. Independence: Errors Are Uncorrelated

**What it means:** $\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$. Knowing today's error tells you nothing about tomorrow's error.

**Why it matters:** If errors are correlated ("autocorrelation" or "serial correlation"), the OLS estimates are still unbiased, but the **standard errors are wrong**. Specifically, with positive autocorrelation (the most common case in finance), standard errors are **understated**, making coefficients appear more significant than they really are.

**What goes wrong:** You think you have found a statistically significant relationship, but it is a mirage caused by correlated errors. You reject the null hypothesis when you should not.

**Financial example:** Monthly returns often exhibit momentum effects — a positive return this month slightly increases the probability of a positive return next month. This creates autocorrelated residuals in return regressions.

> **CFA Exam Tip:** The Durbin-Watson statistic tests for first-order autocorrelation. It ranges from 0 to 4, with a value near 2 indicating no autocorrelation. If DW is significantly below 2, you have positive autocorrelation; significantly above 2, negative autocorrelation.

#### 3. Homoscedasticity: Constant Error Variance

**What it means:** $\text{Var}(\varepsilon_i) = \sigma^2$ for all $i$. The spread of the errors is the same regardless of the value of X.

**Why it matters:** If error variance changes with X ("heteroscedasticity"), the OLS estimates are still unbiased but **no longer efficient** (they are not the minimum-variance estimates). More importantly, the standard errors computed by OLS are wrong.

**What goes wrong:** On a residual plot, you see a "fan shape" — residuals are tightly clustered for small values of X but spread out for large values (or vice versa). Confidence intervals and hypothesis tests become unreliable.

**Financial example:** Volatility clustering in stock returns — periods of high volatility (like 2008) have much larger residuals than calm periods. Regressing daily returns on a factor will often show heteroscedasticity.

> **Common Mistake:** Students often confuse homoscedasticity with normality. They are different assumptions! You can have normally distributed errors that are heteroscedastic (variance changes with X), or non-normal errors that are homoscedastic (constant variance). Always check both.

#### 4. Normality: Errors Follow a Normal Distribution

**What it means:** $\varepsilon_i \sim N(0, \sigma^2)$. The errors follow a bell-shaped curve centred at zero.

**Why it matters:** The t-tests and F-tests used for hypothesis testing are derived under the assumption of normality. If errors are not normal, these tests are only *approximately* valid.

**What goes wrong:** In small samples, non-normal errors can lead to incorrect p-values and confidence intervals. Fortunately, the **Central Limit Theorem** comes to the rescue in large samples — even with non-normal errors, the sampling distributions of the coefficients approach normality as $n$ grows.

**Financial example:** Stock returns often have "fat tails" — extreme returns occur more frequently than the normal distribution predicts. This means residuals in return regressions are often leptokurtic (heavy-tailed), violating normality.

| Assumption | What it means | What goes wrong if violated | Detection method |
|:-----------|:-------------|:---------------------------|:----------------|
| **Linearity** | Relationship is linear | Biased estimates | Residual plot (look for patterns) |
| **Independence** | Errors are uncorrelated | Understated standard errors | Durbin-Watson test |
| **Homoscedasticity** | Error variance is constant | Inefficient estimates, wrong SEs | Residual plot (fan shape), Breusch-Pagan test |
| **Normality** | Errors are normally distributed | Invalid small-sample tests | Q-Q plot, Jarque-Bera test |

---
### Setup

We will use NumPy for matrix operations and implement OLS regression entirely from scratch. No black-box regression libraries — we build every formula ourselves so we understand exactly what is happening under the hood.

The code below imports our dependencies and sets up the plotting style used throughout the notebook.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

# ── Tolerances
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

With our tools ready, let's proceed to build simple linear regression from the ground up. We will start with the mathematical model, derive the OLS formulas step by step, implement them in code, and then validate our implementation.

> **Key Concept:** Throughout this notebook, we use **simulated data with known parameters**. This is a powerful pedagogical technique: because we know the "truth," we can check whether our estimation procedure recovers it correctly. In real-world analysis, you never know the true parameters — but building intuition with simulated data helps you interpret real results with confidence.

The simulation approach also lets us explore important concepts like:
- **Sampling variability:** Different random samples give different estimates.
- **Bias vs precision:** Are the estimates centred on the truth? How spread out are they?
- **The effect of sample size:** More data means more precise estimates.
- **The effect of noise:** Higher idiosyncratic volatility means less precise beta estimates and lower $R^2$.

---
## 1. Simple Linear Regression

### Model

$$y_i = \alpha + \beta x_i + \varepsilon_i, \qquad \varepsilon_i \sim N(0, \sigma^2)$$

where $\alpha$ (intercept) and $\beta$ (slope) are unknown parameters.

**In financial terms:**
- $y_i$ = the stock's return in month $i$
- $x_i$ = the market's return in month $i$
- $\alpha$ = the stock's "alpha" — return earned beyond what market exposure would predict
- $\beta$ = the stock's "beta" — sensitivity to market movements
- $\varepsilon_i$ = the stock's idiosyncratic return in month $i$ — the part of the return specific to this stock

### OLS Derivation

We minimize the **Sum of Squared Errors (SSE)**:

$$\text{SSE} = \sum_{i=1}^{n} (y_i - \alpha - \beta x_i)^2$$

Taking partial derivatives and setting to zero:

$$\frac{\partial \text{SSE}}{\partial \alpha} = -2\sum(y_i - \alpha - \beta x_i) = 0$$

$$\frac{\partial \text{SSE}}{\partial \beta} = -2\sum x_i(y_i - \alpha - \beta x_i) = 0$$

### Closed-Form Solution

From the first equation: $\hat{\alpha} = \bar{y} - \hat{\beta}\bar{x}$

Substituting into the second:

$$\hat{\beta} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2} = \frac{\text{Cov}(x,y)}{\text{Var}(x)}$$

### What the Slope and Intercept Mean

**The slope** $\hat{\beta}$ tells you: **for each 1-unit increase in X, Y changes by $\hat{\beta}$ units, on average.** In a CAPM regression:

- $\hat{\beta} = 1.5$ means "when the market goes up 1%, this stock goes up 1.5% on average" — it amplifies market movements by 50%.
- $\hat{\beta} = 0.5$ means "when the market goes up 1%, this stock goes up only 0.5% on average" — it dampens market movements.
- $\hat{\beta} = -0.3$ means "when the market goes up 1%, this stock goes *down* 0.3% on average" — it moves opposite to the market (like some gold stocks).

**The intercept** $\hat{\alpha}$ tells you: **the expected value of Y when X = 0.** In a CAPM regression, this is Jensen's alpha — the return earned when the market return is zero. A positive alpha suggests the stock is earning more than its risk exposure would predict.

> **Common Mistake:** Do not over-interpret the intercept. In many regressions, $X = 0$ is outside the range of the data, making the intercept economically meaningless. For example, in a regression of return on market cap, a market cap of zero makes no sense — the intercept is just a mathematical anchor for the line.

### Implementing OLS from Scratch

Now let's translate the mathematical derivation into code. We implement the `ols_simple` function using the closed-form formulas derived above, then apply it to simulated CAPM data.

**What the code does:**
1. Defines `ols_simple(x, y)` — computes $\hat{\alpha}$ and $\hat{\beta}$ from scratch using means and cross-products.
2. Generates 60 months of simulated market returns and stock returns using the CAPM model with known parameters ($\alpha = 0.002$, $\beta = 1.3$).
3. Estimates $\hat{\alpha}$ and $\hat{\beta}$ and compares them to the true values.
4. Verifies the computation using the covariance/variance formula.

> **Key Concept:** Because the data includes random noise ($\varepsilon_i$), the OLS estimates will NOT exactly equal the true parameters. They are *estimates* — close to the truth but with some sampling error. If you re-ran the simulation with different random numbers, you would get slightly different estimates. This is the essence of **sampling variability**.

> **What to watch for:** Our from-scratch estimates should match NumPy's output exactly (up to floating-point precision). The estimates should be *close* to the true parameters but not identical.

In [ ]:
def ols_simple(x, y):
    """Ordinary Least Squares for simple linear regression.
    
    Parameters
    ----------
    x, y : 1D arrays of observations
    
    Returns
    -------
    alpha : float – intercept
    beta  : float – slope
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    n = len(x)
    
    x_bar = np.sum(x) / n
    y_bar = np.sum(y) / n
    
    # Slope: Cov(x,y) / Var(x)
    numerator = np.sum((x - x_bar) * (y - y_bar))
    denominator = np.sum((x - x_bar) ** 2)
    
    beta = numerator / denominator
    alpha = y_bar - beta * x_bar
    
    return alpha, beta


# ── Generate sample data: stock return vs market return
n_obs = 60  # 60 monthly observations
true_alpha = 0.002   # 0.2% monthly alpha
true_beta = 1.3      # aggressive stock
sigma_eps = 0.03     # idiosyncratic volatility

# Market returns (simulate realistic distribution)
market_returns = rng.normal(0.008, 0.045, n_obs)

# Stock returns from CAPM + noise
stock_returns = true_alpha + true_beta * market_returns + rng.normal(0, sigma_eps, n_obs)

# Estimate
alpha_hat, beta_hat = ols_simple(market_returns, stock_returns)

print(f"OLS Regression: R_stock = α + β·R_market + ε")
print(f"\n  True parameters:  α = {true_alpha:.4f}, β = {true_beta:.4f}")
print(f"  OLS estimates:    α̂ = {alpha_hat:.4f}, β̂ = {beta_hat:.4f}")

# ── Verify against manual formula
cov_xy = np.sum((market_returns - np.mean(market_returns)) * 
                (stock_returns - np.mean(stock_returns))) / (n_obs - 1)
var_x = np.sum((market_returns - np.mean(market_returns))**2) / (n_obs - 1)
beta_verify = cov_xy / var_x
print(f"\n  Verification: Cov(x,y)/Var(x) = {cov_xy:.6f}/{var_x:.6f} = {beta_verify:.4f}")

### Interpreting the OLS Output

Look at the output above. The key things to notice:

1. **The estimates are close to but not equal to the true values.** This is expected! With only 60 observations and noise ($\sigma_\varepsilon = 0.03$), we get sampling error. With more data, the estimates would converge to the true values — this is the property of **consistency**.

2. **The verification confirms our formula.** $\hat{\beta} = \text{Cov}(x,y) / \text{Var}(x)$ gives the same answer as computing it from deviations. There is a subtle difference: the verification uses $n-1$ in the denominator (sample covariance/variance), but because $n-1$ appears in both numerator and denominator, it cancels out.

3. **The intercept $\hat{\alpha}$** is the estimated Jensen's alpha. If close to zero, the stock is earning roughly what CAPM predicts. If significantly positive, the stock may be "beating the market" on a risk-adjusted basis.

> **CFA Exam Tip:** When interpreting regression output on the exam, always state the economic meaning. Do NOT just say "the slope is 1.3." Instead say: "The estimated beta is 1.3, meaning that for every 1% increase in the market return, the stock's return is expected to increase by 1.3%, on average."

### Visualising the Regression

The scatter plot below shows each month as a dot (market return on the X-axis, stock return on the Y-axis). The **coral line** is our OLS estimate, and the **green dashed line** is the true relationship.

**What to look for:**
- The dots should be scattered roughly evenly above and below the line (this means the residuals are balanced).
- The OLS line (coral) should be close to the true line (green) but not identical — the gap is sampling error.
- The vertical distance from each dot to the coral line is the **residual** $e_i = y_i - \hat{y}_i$. OLS minimises the sum of squares of these distances.

In a CAPM context, if X is market excess return and Y is stock excess return, the slope of this line IS the stock's beta.

In [ ]:
# ── Visualization: Scatter plot with regression line
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(market_returns * 100, stock_returns * 100, color=PRIMARY, alpha=0.6, 
           edgecolor='white', s=60, label='Observed')

# Regression line
x_line = np.linspace(market_returns.min(), market_returns.max(), 100)
y_line = alpha_hat + beta_hat * x_line
ax.plot(x_line * 100, y_line * 100, color=SECONDARY, linewidth=2.5,
        label=f'OLS: ŷ = {alpha_hat:.4f} + {beta_hat:.2f}x')

# True line for comparison
y_true = true_alpha + true_beta * x_line
ax.plot(x_line * 100, y_true * 100, color=TERTIARY, linewidth=2, linestyle='--',
        label=f'True: y = {true_alpha:.4f} + {true_beta:.2f}x')

ax.set_xlabel('Market Return (%)')
ax.set_ylabel('Stock Return (%)')
ax.set_title('Simple Linear Regression: Stock vs Market Returns')
ax.legend()
plt.tight_layout()
plt.show()

### Reading the Scatter Plot

The plot above is the foundational visualisation of regression. Notice how:

- The dots form a **positive cloud** sloping upward from left to right, confirming the positive relationship between market and stock returns.
- The OLS line (coral) passes through the **centre of the cloud**. This is not a coincidence — the OLS line always passes through $(\bar{X}, \bar{Y})$.
- Some dots are well above the line (the stock outperformed what the market alone would predict) and some are well below (the stock underperformed). These deviations are the **residuals**.
- The gap between the OLS line and the true line represents **estimation error** due to finite sample size and noise.

> **Key Concept:** The scatter plot is the most important single visual in regression analysis. Before computing anything, always look at the scatter plot. It can reveal nonlinearity, outliers, and heteroscedasticity that summary statistics might miss.

---
## 2. Regression Diagnostics

Fitting a line is only the first step. The next — and arguably more important — step is asking: **how good is the fit, and can we trust the results?**

### Coefficient of Determination ($R^2$)

$$R^2 = 1 - \frac{\text{SSE}}{\text{SST}} = \frac{\text{SSR}}{\text{SST}}$$

where:
- $\text{SST} = \sum(y_i - \bar{y})^2$ — Total Sum of Squares (total variation in Y)
- $\text{SSR} = \sum(\hat{y}_i - \bar{y})^2$ — Regression Sum of Squares (variation explained by the line)
- $\text{SSE} = \sum(y_i - \hat{y}_i)^2$ — Error Sum of Squares (variation NOT explained by the line)

Notice that $\text{SST} = \text{SSR} + \text{SSE}$. This is the ANOVA decomposition — every bit of variation is either "explained" or "unexplained."

### Adjusted $R^2$: Penalising for Complexity

$$\bar{R}^2 = 1 - \frac{(1-R^2)(n-1)}{n - k - 1}$$

where $k$ is the number of predictors. Adjusted $R^2$ penalizes for adding variables that do not genuinely improve the model.

**Why do we need this?** Regular $R^2$ can *never decrease* when you add another variable — even if that variable is pure noise. Adjusted $R^2$ can decrease, making it a better tool for comparing models with different numbers of predictors.

> **CFA Exam Tip:** When comparing multiple regression models with different numbers of independent variables, always use adjusted $R^2$, not regular $R^2$. A model with a higher adjusted $R^2$ is preferred.

### Standard Error of the Estimate (SEE)

$$\text{SEE} = \sqrt{\frac{\text{SSE}}{n - 2}}$$

The SEE measures the **average size of the residuals** — it is the standard deviation of the regression errors. A smaller SEE means a tighter fit. In financial terms, if SEE = 3%, then on average, the stock's actual return differs from the regression prediction by about 3 percentage points.

> **Key Concept:** SEE is in the **same units as Y**. If Y is measured in percent, SEE is in percent. This makes it directly interpretable: "the regression's predictions are typically off by about SEE."

### Computing Diagnostics from Scratch

The code below implements the `regression_diagnostics` function. It computes:

- Predicted values $\hat{y}_i$ and residuals $e_i = y_i - \hat{y}_i$
- The three sums of squares: SST, SSR, SSE
- $R^2$, adjusted $R^2$, and SEE

It then verifies the ANOVA identity $\text{SST} = \text{SSR} + \text{SSE}$ — if this does not hold, something is wrong with our calculation.

> **What to watch for:** The $R^2$ value tells us how much of the stock return variation is explained by the market. For a stock with true $\beta = 1.3$ and idiosyncratic volatility of 3%, we would expect a moderate-to-high $R^2$.

In [ ]:
def regression_diagnostics(x, y, alpha, beta):
    """Compute regression diagnostics from scratch.
    
    Returns dict with: y_hat, residuals, SST, SSR, SSE, R2, adj_R2, SEE
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    n = len(x)
    k = 1  # one predictor
    
    y_hat = alpha + beta * x
    residuals = y - y_hat
    y_bar = np.mean(y)
    
    SST = np.sum((y - y_bar) ** 2)
    SSR = np.sum((y_hat - y_bar) ** 2)
    SSE = np.sum(residuals ** 2)
    
    R2 = 1 - SSE / SST
    adj_R2 = 1 - (1 - R2) * (n - 1) / (n - k - 1)
    SEE = np.sqrt(SSE / (n - 2))
    
    return {
        'y_hat': y_hat,
        'residuals': residuals,
        'SST': SST,
        'SSR': SSR,
        'SSE': SSE,
        'R2': R2,
        'adj_R2': adj_R2,
        'SEE': SEE,
    }


diag = regression_diagnostics(market_returns, stock_returns, alpha_hat, beta_hat)

print("Regression Diagnostics:")
print(f"  SST = {diag['SST']:.6f}")
print(f"  SSR = {diag['SSR']:.6f}")
print(f"  SSE = {diag['SSE']:.6f}")
print(f"  SST = SSR + SSE? {np.isclose(diag['SST'], diag['SSR'] + diag['SSE'])}")
print(f"\n  R²       = {diag['R2']:.4f}  ({diag['R2']:.1%} of variance explained)")
print(f"  Adj R²   = {diag['adj_R2']:.4f}")
print(f"  SEE      = {diag['SEE']:.4f}")

### Interpreting the Diagnostics

Let's unpack what the output tells us:

- **SST = SSR + SSE:** This identity holds (the check says `True`). This is a mathematical certainty for OLS — if it failed, we would have a bug.
- **$R^2$:** The market return explains a substantial fraction of the stock's return variation. For an aggressive stock ($\beta = 1.3$) with moderate idiosyncratic noise, this is expected.
- **Adjusted $R^2$:** Slightly lower than $R^2$, as expected. The penalty is small because we have 60 observations and only 1 predictor.
- **SEE:** This is roughly the standard deviation of the residuals. It tells us the typical prediction error in absolute terms.

> **Key Concept:** $R^2$ is a **relative** measure (what fraction of variation is explained), while SEE is an **absolute** measure (how large are the typical errors). Both are useful. A high $R^2$ with a large SEE means the model explains the pattern well but individual predictions are still noisy — common in financial applications.

### Visualising Residuals: The Single Most Important Diagnostic

Numbers alone do not tell the full story. **Residual plots** are the most powerful diagnostic tool because they can reveal problems that summary statistics miss.

The code below creates three diagnostic plots:

1. **Residuals vs Fitted Values:** This is the workhorse diagnostic plot. If the regression assumptions hold, this should look like a random cloud of points with no discernible pattern.
2. **Histogram of Residuals:** Should be roughly bell-shaped and centred at zero (testing normality).
3. **Q-Q Plot:** Quantile-quantile plot comparing the residual distribution to a theoretical normal distribution. Points should fall close to the 45-degree line.

> **What to look for:**
> - **Residuals vs fitted:** Should look like random scatter (no patterns). A U-shape suggests nonlinearity. A fan shape suggests heteroscedasticity.
> - **Histogram:** Should be roughly symmetric and bell-shaped.
> - **Q-Q plot:** Points should follow the diagonal line. Departures at the tails suggest heavy tails (leptokurtosis) — common in financial data.

> **CFA Exam Tip:** You will not be asked to draw these plots on the exam, but you WILL be asked to interpret them. Know what "good" vs "bad" residual patterns look like.

In [ ]:
# ── Residual plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs fitted values
axes[0].scatter(diag['y_hat'] * 100, diag['residuals'] * 100, 
                color=PRIMARY, alpha=0.6, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Fitted Values (%)')
axes[0].set_ylabel('Residuals (%)')
axes[0].set_title('Residuals vs Fitted')

# Histogram of residuals
axes[1].hist(diag['residuals'] * 100, bins=20, density=True, 
             color=PRIMARY, alpha=0.7, edgecolor='white')
x_norm = np.linspace(-10, 10, 100)
axes[1].plot(x_norm, stats.norm.pdf(x_norm, 0, diag['SEE'] * 100), 
             color=SECONDARY, linewidth=2, label='Normal fit')
axes[1].set_xlabel('Residual (%)')
axes[1].set_ylabel('Density')
axes[1].set_title('Residual Distribution')
axes[1].legend()

# QQ plot
sorted_resid = np.sort(diag['residuals'])
n_r = len(sorted_resid)
theoretical = stats.norm.ppf(np.arange(1, n_r + 1) / (n_r + 1))
axes[2].scatter(theoretical, sorted_resid / diag['SEE'], 
                color=PRIMARY, alpha=0.6, edgecolor='white')
axes[2].plot([-3, 3], [-3, 3], color=SECONDARY, linewidth=2, linestyle='--')
axes[2].set_xlabel('Theoretical Quantiles')
axes[2].set_ylabel('Standardized Residuals')
axes[2].set_title('Q-Q Plot')

plt.tight_layout()
plt.show()

### Reading the Residual Plots

**Residuals vs Fitted (left panel):**
- The points appear randomly scattered around zero with no obvious pattern — this is exactly what we want.
- No fan shape (heteroscedasticity) or curvature (nonlinearity) is visible.
- This supports the assumptions of linearity and homoscedasticity.

**Histogram (centre panel):**
- The histogram is roughly bell-shaped and centred near zero.
- The orange line shows the theoretical normal distribution with the same standard deviation.
- A reasonable match supports the normality assumption.

**Q-Q Plot (right panel):**
- Points fall close to the 45-degree line, confirming approximate normality.
- Mild departures at the extreme tails are common and usually not a concern with 60 observations.

> **Key Concept:** Our simulated data was generated from a normal distribution, so it *should* pass these checks. With real financial data, you will often see heavier tails (more extreme values than the normal predicts), slight skewness, and possibly heteroscedasticity. That is when the assumption diagnostics become truly valuable.

> **Common Mistake:** Do not confuse "residuals look roughly normal" with "all assumptions are satisfied." Normality is just one of four assumptions. You still need to check for autocorrelation (Durbin-Watson), heteroscedasticity (Breusch-Pagan), and nonlinearity (residual patterns).

---
## 3. Hypothesis Tests for Coefficients

We have estimated $\hat{\alpha}$ and $\hat{\beta}$. But how do we know if these estimates are **statistically meaningful**? Maybe the true slope is actually zero (no relationship between the stock and the market), and we just got an estimated slope of 1.3 by sheer luck?

This is the question hypothesis testing answers.

### The Big Question: Is the Slope Really Different from Zero?

We set up the hypotheses:

- $H_0: \beta = 0$ (there is NO linear relationship between X and Y)
- $H_a: \beta \neq 0$ (there IS a linear relationship)

If we cannot reject $H_0$, we cannot claim that X helps predict Y. In a CAPM context, failing to reject $H_0$ for beta would mean we cannot establish that the stock has any systematic market exposure — a serious conclusion.

### T-Test for Individual Coefficients

The test statistic is:

$$t = \frac{\hat{\beta}}{\text{SE}(\hat{\beta})} \sim t_{n-2}$$

where the standard error of the slope is:

$$\text{SE}(\hat{\beta}) = \frac{\text{SEE}}{\sqrt{\sum(x_i - \bar{x})^2}}$$

And the standard error of the intercept is:

$$\text{SE}(\hat{\alpha}) = \text{SEE} \cdot \sqrt{\frac{1}{n} + \frac{\bar{x}^2}{\sum(x_i - \bar{x})^2}}$$

**Intuition behind the t-statistic:**

The t-statistic is a **signal-to-noise ratio**. The numerator $\hat{\beta}$ is the signal (how large is the estimated effect?). The denominator $\text{SE}(\hat{\beta})$ is the noise (how precise is the estimate?). A large t-statistic means the signal is much larger than the noise — the estimate is reliably different from zero.

> **CFA Exam Tip:** A quick rule of thumb: if $|t| > 2$, the coefficient is significant at the 5% level (approximately, for large samples). This works because for $n > 30$, the critical value of the t-distribution is close to 1.96.

### Confidence Interval for the Slope

$$\hat{\beta} \pm t_{\alpha/2, n-2} \cdot \text{SE}(\hat{\beta})$$

A 95% confidence interval means: if we repeated this analysis many times with different samples from the same population, about 95% of the intervals would contain the true $\beta$. It does NOT mean there is a 95% probability that the true $\beta$ is in this particular interval (a common misinterpretation).

> **Common Mistake:** The confidence interval is about the *procedure*, not about any single interval. Saying "there is a 95% chance the true beta is between 1.0 and 1.6" is technically incorrect. The correct interpretation is: "We are 95% confident that the true beta is between 1.0 and 1.6, meaning that if we repeated this study many times, 95% of the resulting intervals would contain the true beta."

### F-Test for Overall Significance

$$F = \frac{\text{SSR}/k}{\text{SSE}/(n-k-1)} = \frac{\text{MSR}}{\text{MSE}} \sim F_{k, n-k-1}$$

The F-test asks: **does the regression as a whole explain a significant amount of variation?** For simple regression with one predictor, the F-test is equivalent to the t-test for the slope (in fact, $F = t^2$). But in multiple regression, the F-test becomes distinct — it tests whether *all* the predictors jointly are significant.

> **Key Concept:** The t-test tests individual coefficients one at a time. The F-test tests all coefficients simultaneously. In multiple regression, it is possible for no individual coefficient to be significant (all t-tests fail), yet the F-test is significant (the variables jointly matter). This can happen when the predictors are highly correlated with each other (multicollinearity).

### Computing Hypothesis Tests from Scratch

The code below implements all the hypothesis testing machinery:

1. Standard errors for both $\hat{\alpha}$ and $\hat{\beta}$
2. t-statistics and p-values for each coefficient
3. A 95% confidence interval for $\hat{\beta}$
4. The F-statistic and its p-value
5. Verification that $F = t^2$ (which should hold in simple regression)

> **What to watch for:** We check whether the true $\beta = 1.3$ falls inside the 95% confidence interval. If our code is correct and the sample is not pathological, it should.

In [ ]:
def coefficient_tests(x, y, alpha_coef, beta_coef, diagnostics):
    """Perform hypothesis tests on regression coefficients.
    
    Returns dict with t-stats, p-values, standard errors, and F-stat.
    """
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    n = len(x)
    k = 1
    SEE = diagnostics['SEE']
    
    x_bar = np.mean(x)
    SS_xx = np.sum((x - x_bar) ** 2)
    
    # Standard errors
    se_beta = SEE / np.sqrt(SS_xx)
    se_alpha = SEE * np.sqrt(1/n + x_bar**2 / SS_xx)
    
    # T-statistics
    t_beta = beta_coef / se_beta
    t_alpha = alpha_coef / se_alpha
    
    # P-values (two-sided)
    df = n - 2
    p_beta = 2 * (1 - stats.t.cdf(abs(t_beta), df))
    p_alpha = 2 * (1 - stats.t.cdf(abs(t_alpha), df))
    
    # F-statistic
    MSR = diagnostics['SSR'] / k
    MSE = diagnostics['SSE'] / (n - k - 1)
    F_stat = MSR / MSE
    p_F = 1 - stats.f.cdf(F_stat, k, n - k - 1)
    
    # Confidence intervals for beta
    t_crit = stats.t.ppf(0.975, df)
    beta_ci = (beta_coef - t_crit * se_beta, beta_coef + t_crit * se_beta)
    
    return {
        'se_alpha': se_alpha, 'se_beta': se_beta,
        't_alpha': t_alpha, 't_beta': t_beta,
        'p_alpha': p_alpha, 'p_beta': p_beta,
        'F_stat': F_stat, 'p_F': p_F,
        'beta_ci': beta_ci,
        'df': df,
    }


tests = coefficient_tests(market_returns, stock_returns, alpha_hat, beta_hat, diag)

print("Coefficient Tests (df = {}):\n".format(tests['df']))
print(f"{'Coef':<10} {'Estimate':>10} {'Std Error':>10} {'t-stat':>10} {'p-value':>10}")
print("-" * 52)
print(f"{'α (int)':.<10} {alpha_hat:>10.4f} {tests['se_alpha']:>10.4f} "
      f"{tests['t_alpha']:>10.4f} {tests['p_alpha']:>10.4f}")
print(f"{'β (slope)':.<10} {beta_hat:>10.4f} {tests['se_beta']:>10.4f} "
      f"{tests['t_beta']:>10.4f} {tests['p_beta']:>10.4f}")

print(f"\n95% CI for β: [{tests['beta_ci'][0]:.4f}, {tests['beta_ci'][1]:.4f}]")
print(f"True β = {true_beta:.4f} in CI? {tests['beta_ci'][0] <= true_beta <= tests['beta_ci'][1]}")

print(f"\nF-statistic = {tests['F_stat']:.4f}, p-value = {tests['p_F']:.6f}")
print(f"t²(β) = {tests['t_beta']**2:.4f} ≈ F = {tests['F_stat']:.4f}")

### Interpreting the Hypothesis Test Results

Let's walk through each piece of the output:

**The slope ($\hat{\beta}$):**
- The t-statistic is large (much greater than 2), and the p-value is very small (essentially zero). We overwhelmingly reject $H_0: \beta = 0$.
- **Conclusion:** There is a statistically significant linear relationship between market returns and stock returns. The stock has genuine market exposure.
- The 95% confidence interval contains the true value $\beta = 1.3$, confirming that our estimate is consistent with the data-generating process.

**The intercept ($\hat{\alpha}$):**
- The t-statistic for the intercept may or may not be significant. In CAPM regressions, alpha is often small and statistically insignificant, which is what CAPM predicts (no free lunch).
- If the p-value for alpha is above 0.05, we cannot reject $H_0: \alpha = 0$, meaning we have no evidence of abnormal return.

**F-statistic:**
- The F-statistic equals $t^2$ for the slope — this is a mathematical identity in simple regression.
- The significant F-statistic confirms the regression as a whole is meaningful.

> **CFA Exam Tip:** When presented with regression output on the exam, follow this checklist:
> 1. Is the F-statistic significant? (If not, the entire regression is questionable.)
> 2. Which individual coefficients are significant? (Check t-statistics or p-values.)
> 3. What is $R^2$? (How much variation is explained?)
> 4. Interpret the coefficients in economic terms. (What do the numbers MEAN?)

> **Common Mistake:** Students sometimes test $H_0: \beta = 1$ (is the stock a "market stock"?) but use the t-statistic for $H_0: \beta = 0$. To test $H_0: \beta = 1$, you need $t = (\hat{\beta} - 1) / \text{SE}(\hat{\beta})$, NOT $t = \hat{\beta} / \text{SE}(\hat{\beta})$. The null hypothesis value changes the numerator.

---
## 4. ANOVA Table

The **Analysis of Variance** table is the standard way to present the decomposition of total variation. It puts all the key quantities in one place.

### The Decomposition

$$\text{SST} = \text{SSR} + \text{SSE}$$

| Source | df | SS | MS | F |
|---|---|---|---|---|
| Regression | $k$ | SSR | MSR = SSR/$k$ | MSR/MSE |
| Error | $n-k-1$ | SSE | MSE = SSE/$(n-k-1)$ | |
| Total | $n-1$ | SST | | |

**Reading the ANOVA table:**

- **Degrees of freedom (df):** The regression "uses up" $k$ degrees of freedom (one for each predictor). The error has $n - k - 1$ degrees of freedom remaining. The total is $n - 1$.
- **Mean Square (MS):** This is SS divided by df — it is the "average" variation per degree of freedom. MSR is the average explained variation per predictor; MSE is the average unexplained variation per observation (minus parameters estimated).
- **F-statistic:** MSR/MSE. If the regression explains nothing, MSR and MSE should be about equal ($F \approx 1$). A large F means the regression explains far more per degree of freedom than noise alone.

> **Key Concept:** The ANOVA table is a compact summary of the regression's explanatory power. The F-test at the bottom answers the fundamental question: **is this regression worth anything?**

### Building the ANOVA Table from Scratch

The code below constructs and prints a complete ANOVA table. We verify that the degrees of freedom add up ($k + (n-k-1) = n-1$) and that the F-statistic matches what we computed earlier.

> **What to look for:** A large F-statistic (and small p-value) means the regression explains a statistically significant portion of the variation in Y. The F-test p-value should match what we found in the hypothesis testing section.

In [ ]:
def anova_table(x, y, diagnostics, k=1):
    """Build ANOVA table for regression."""
    n = len(x)
    SSR = diagnostics['SSR']
    SSE = diagnostics['SSE']
    SST = diagnostics['SST']
    
    df_reg = k
    df_err = n - k - 1
    df_tot = n - 1
    
    MSR = SSR / df_reg
    MSE = SSE / df_err
    F = MSR / MSE
    p_F = 1 - stats.f.cdf(F, df_reg, df_err)
    
    return {
        'sources': ['Regression', 'Error', 'Total'],
        'df': [df_reg, df_err, df_tot],
        'SS': [SSR, SSE, SST],
        'MS': [MSR, MSE, None],
        'F': [F, None, None],
        'p': [p_F, None, None],
    }


anova = anova_table(market_returns, stock_returns, diag)

print(f"{'Source':<12} {'df':>4} {'SS':>12} {'MS':>12} {'F':>10} {'p-value':>10}")
print("-" * 62)
for i in range(3):
    ms_str = f"{anova['MS'][i]:.6f}" if anova['MS'][i] is not None else ''
    f_str = f"{anova['F'][i]:.4f}" if anova['F'][i] is not None else ''
    p_str = f"{anova['p'][i]:.6f}" if anova['p'][i] is not None else ''
    print(f"{anova['sources'][i]:<12} {anova['df'][i]:>4} {anova['SS'][i]:>12.6f} "
          f"{ms_str:>12} {f_str:>10} {p_str:>10}")

### Reading the ANOVA Output

The ANOVA table above shows:

- **Regression row:** The SSR and its mean square tell us how much variation the market return explains. The large F-statistic and tiny p-value confirm the regression is highly significant.
- **Error row:** The SSE and MSE tell us how much variation remains unexplained. MSE is the estimate of $\sigma^2$ (the error variance).
- **Total row:** SST is the total variation before regression.

Notice that $R^2 = \text{SSR} / \text{SST}$. You can verify this from the numbers in the table.

> **CFA Exam Tip:** You may be asked to calculate the F-statistic from an ANOVA table, or to fill in missing values given partial information. Remember: SST = SSR + SSE, df always add up, and MS = SS/df. These three relationships let you reconstruct any missing piece.

### Connecting ANOVA to R-squared

The ANOVA table and $R^2$ are two sides of the same coin:

$$R^2 = \frac{\text{SSR}}{\text{SST}} = 1 - \frac{\text{SSE}}{\text{SST}}$$

You can always compute $R^2$ from an ANOVA table by dividing the Regression SS by the Total SS. This is a useful shortcut on the exam.

Similarly, the F-statistic can be expressed in terms of $R^2$:

$$F = \frac{R^2 / k}{(1 - R^2) / (n - k - 1)}$$

This formula shows that the F-statistic increases with $R^2$ (better fit) and with $n$ (more data). Even a modest $R^2$ can be highly significant with enough data.

**Worked example:** Suppose $R^2 = 0.25$, $k = 1$, and $n = 50$:

$$F = \frac{0.25 / 1}{0.75 / 48} = \frac{0.25}{0.0156} = 16.0$$

With $F_{1,48} = 16.0$, the p-value is well below 0.001. So even though the regression explains only 25% of the variation, the relationship is highly significant with 50 observations.

> **CFA Exam Tip:** You should be able to go back and forth between the ANOVA table, $R^2$, and the F-statistic. Given any two of these, you can compute the third. This is a frequent exam question format: "Given the following partial ANOVA table, compute the missing values."

---
## 5. Assumptions and Violations

The classical linear regression model assumes:

1. **Linearity:** $E[y|x]$ is linear in $x$
2. **Homoscedasticity:** $\text{Var}(\varepsilon_i) = \sigma^2$ (constant)
3. **Normality:** $\varepsilon_i \sim N(0, \sigma^2)$
4. **Independence:** $\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$

We discussed earlier *why* each matters. Now let's implement **formal statistical tests** to detect violations.

### The Durbin-Watson Test for Autocorrelation

$$DW = \frac{\sum_{t=2}^{n}(e_t - e_{t-1})^2}{\sum_{t=1}^{n}e_t^2} \approx 2(1 - \hat{\rho})$$

where $\hat{\rho}$ is the first-order autocorrelation of the residuals.

**Interpretation:**
- $DW \approx 2$: No autocorrelation ($\hat{\rho} \approx 0$) — this is what we want.
- $DW \approx 0$: Strong positive autocorrelation ($\hat{\rho} \approx 1$) — consecutive residuals tend to have the same sign.
- $DW \approx 4$: Strong negative autocorrelation ($\hat{\rho} \approx -1$) — consecutive residuals tend to flip sign.

**Why does autocorrelation matter in finance?** Financial time series often exhibit autocorrelation due to:
- **Momentum effects:** Past returns predict future returns.
- **Slow information incorporation:** News gets priced in gradually, not instantly.
- **Stale pricing:** Some assets (real estate, private equity) are not priced continuously.

When residuals are autocorrelated, standard errors are biased downward, making everything look more significant than it truly is.

### The Breusch-Pagan Test for Heteroscedasticity

**Idea:** Regress the squared residuals $e_i^2$ on $x_i$ and test if the slope is significant. If it is, the variance of the errors changes with $x$ — that is heteroscedasticity.

**Why does heteroscedasticity matter in finance?** Volatility clustering is pervasive in financial markets. During crises, both market and stock returns become more volatile, producing larger residuals. Standard OLS confidence intervals become unreliable.

> **CFA Exam Tip:** If you detect heteroscedasticity, the fix is to use **robust standard errors** (also called White's standard errors or heteroscedasticity-consistent standard errors). The coefficient estimates remain the same; only the standard errors (and hence t-statistics and p-values) change.

### Implementing the Tests

The code below implements the Durbin-Watson and Breusch-Pagan tests from scratch and applies them to our regression residuals.

> **Practical note:** In finance, autocorrelation (serial correlation) and heteroscedasticity are extremely common in time-series data. Don't just assume the standard OLS results are valid — always test! If the tests reveal violations, you need to either fix the model (e.g., add lagged variables to address autocorrelation) or use robust methods (e.g., Newey-West standard errors).

In [ ]:
def durbin_watson(residuals):
    """Durbin-Watson statistic for autocorrelation."""
    e = np.asarray(residuals)
    diff_sq = np.sum((e[1:] - e[:-1]) ** 2)
    sum_sq = np.sum(e ** 2)
    return diff_sq / sum_sq


def breusch_pagan_test(x, residuals):
    """Simplified Breusch-Pagan test for heteroscedasticity.
    Regress e² on x and test if slope is significant.
    """
    e_sq = residuals ** 2
    alpha_bp, beta_bp = ols_simple(x, e_sq)
    diag_bp = regression_diagnostics(x, e_sq, alpha_bp, beta_bp)
    
    n = len(x)
    se_beta_bp = diag_bp['SEE'] / np.sqrt(np.sum((x - np.mean(x))**2))
    t_stat = beta_bp / se_beta_bp
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))
    
    return {'t_stat': t_stat, 'p_value': p_value, 'R2': diag_bp['R2']}


# ── Test our regression residuals
dw = durbin_watson(diag['residuals'])
bp = breusch_pagan_test(market_returns, diag['residuals'])

print("Assumption Tests:\n")
print(f"Durbin-Watson statistic: {dw:.4f}")
if 1.5 < dw < 2.5:
    print(f"  → No significant autocorrelation (DW ≈ 2)")
elif dw < 1.5:
    print(f"  → Positive autocorrelation detected")
else:
    print(f"  → Negative autocorrelation detected")

print(f"\nBreusch-Pagan test: t = {bp['t_stat']:.4f}, p = {bp['p_value']:.4f}")
if bp['p_value'] > 0.05:
    print(f"  → No significant heteroscedasticity at 5% level")
else:
    print(f"  → Heteroscedasticity detected at 5% level")

### Interpreting the Assumption Tests

**Durbin-Watson:**
- A DW statistic near 2 indicates no significant autocorrelation, which is what we expect from our simulated data (the errors were generated independently).
- In practice, with real financial data, you might see DW values significantly below 2 — especially with daily or weekly data where momentum or mean-reversion effects create serial correlation in returns.

**Breusch-Pagan:**
- A non-significant result (p > 0.05) means we cannot reject the null hypothesis of homoscedasticity. Again, expected for our well-behaved simulated data.
- Real-world financial data often fails this test due to volatility clustering.

> **Key Concept:** Our simulated data passes both tests because we *designed* it to satisfy the assumptions. The real value of these tests emerges when you apply them to messy real-world data, where violations are the norm rather than the exception.

### Visualising What Assumption Violations Look Like

To build your intuition, the next plot compares three scenarios:

1. **Good residuals** (our data): Random scatter, constant spread, no pattern.
2. **Heteroscedastic residuals:** The spread of residuals increases with the fitted value, creating a "fan" or "cone" shape. This is what volatility clustering looks like in a residual plot.
3. **Autocorrelated residuals:** Consecutive residuals tend to have the same sign, creating "runs" of positive and negative values. The residual plot looks like a smooth wave rather than random scatter.

Learning to recognise these patterns visually is an essential skill for any analyst.

> **CFA Exam Tip:** On the exam, you might see a residual plot and be asked: "Which assumption is most likely violated?" If you see a fan shape, say heteroscedasticity. If you see a clear pattern of runs (positive residuals clustered together), say autocorrelation. If you see a U-shape or curve, say nonlinearity.

In [ ]:
# ── Illustration of assumption violations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Good residuals (our data)
axes[0].scatter(diag['y_hat'], diag['residuals'], color=TERTIARY, alpha=0.6, edgecolor='white')
axes[0].axhline(0, color='black')
axes[0].set_title(f'Good: Homoscedastic\nDW={dw:.2f}')
axes[0].set_xlabel('Fitted'); axes[0].set_ylabel('Residual')

# Heteroscedastic example
x_het = rng.uniform(0, 10, 100)
y_het = 2 + 3 * x_het + rng.normal(0, 1, 100) * x_het  # variance increases with x
a_het, b_het = ols_simple(x_het, y_het)
resid_het = y_het - (a_het + b_het * x_het)
axes[1].scatter(a_het + b_het * x_het, resid_het, color=SECONDARY, alpha=0.6, edgecolor='white')
axes[1].axhline(0, color='black')
axes[1].set_title('Bad: Heteroscedastic\n(Fan-shaped residuals)')
axes[1].set_xlabel('Fitted'); axes[1].set_ylabel('Residual')

# Autocorrelated example
n_ac = 100
x_ac = np.arange(n_ac)
eps_ac = np.zeros(n_ac)
eps_ac[0] = rng.normal()
for t in range(1, n_ac):
    eps_ac[t] = 0.85 * eps_ac[t-1] + rng.normal()  # AR(1) errors
y_ac = 1 + 0.05 * x_ac + eps_ac
a_ac, b_ac = ols_simple(x_ac, y_ac)
resid_ac = y_ac - (a_ac + b_ac * x_ac)
dw_ac = durbin_watson(resid_ac)
axes[2].plot(x_ac, resid_ac, 'o-', color=PRIMARY, alpha=0.6, markersize=4)
axes[2].axhline(0, color='black')
axes[2].set_title(f'Bad: Autocorrelated\nDW={dw_ac:.2f}')
axes[2].set_xlabel('Time'); axes[2].set_ylabel('Residual')

plt.tight_layout()
plt.show()

### Reading the Assumption Violation Plots

**Left panel (Good):** This is our CAPM regression. The residuals are randomly scattered around zero with roughly constant spread. No patterns, no fan shape, no runs. The DW statistic is near 2. This is what "clean" residuals look like.

**Centre panel (Heteroscedastic):** Notice the clear fan shape — residuals are small for low fitted values and large for high fitted values. This happens because we generated the data with $\varepsilon_i \propto x_i$ (error variance increases with X). In finance, this pattern appears when volatility scales with the level of the variable (e.g., larger stocks have larger absolute price changes).

**Right panel (Autocorrelated):** The residuals follow a smooth wave — when one residual is positive, the next tends to be positive too (and vice versa). The connected lines make the "runs" visually obvious. The DW statistic is well below 2, confirming strong positive autocorrelation. In finance, this pattern appears in return regressions where you have omitted a trending variable.

> **Common Mistake:** Looking at the heteroscedastic plot, some students think the regression line is "wrong" because the residuals are large on the right. The line is actually still the best straight line through the data! The problem is not with the *coefficients* — they are still unbiased. The problem is with the *standard errors* — they do not account for the changing spread, so our t-tests and confidence intervals are unreliable.

### Remedies for Assumption Violations

Detecting a violation is only half the battle. Here is what to do about each:

#### If You Find Nonlinearity
- **Transform the variables:** Try $\log(X)$, $\sqrt{X}$, or $X^2$. In finance, log returns are often more linear than simple returns.
- **Add polynomial terms:** Include $X^2$ or $X^3$ as additional predictors.
- **Use a different model:** Nonlinear regression, splines, or machine learning methods.

#### If You Find Autocorrelation
- **Add lagged variables:** Include $Y_{t-1}$ or $X_{t-1}$ as predictors. This often captures the serial dependence.
- **Use Newey-West standard errors:** These adjust the standard errors for autocorrelation without changing the coefficient estimates.
- **First-difference the data:** Instead of regressing $Y_t$ on $X_t$, regress $\Delta Y_t = Y_t - Y_{t-1}$ on $\Delta X_t$.

#### If You Find Heteroscedasticity
- **Use White's robust standard errors:** The most common fix. The coefficient estimates stay the same; only the standard errors change.
- **Use Weighted Least Squares (WLS):** Give less weight to observations with higher variance.
- **Transform the dependent variable:** Taking logs often stabilises variance (because $\text{Var}(\log X) \approx \text{Var}(X) / E[X]^2$).

#### If You Find Non-normality
- **Increase sample size:** The Central Limit Theorem ensures that t-tests are approximately valid for large $n$, even without normality.
- **Use bootstrap methods:** Resample the data to construct confidence intervals without assuming normality.
- **Check for outliers:** Extreme observations can distort the residual distribution. Investigate whether they are data errors or genuine extreme events.

> **Key Concept:** In practice, the most important violations in finance are **heteroscedasticity** and **autocorrelation**. Robust standard errors (Newey-West for time series, White for cross-sections) are the standard fix. Most professional financial software reports robust standard errors by default.

> **CFA Exam Tip:** The exam focuses on *detection* (Durbin-Watson for autocorrelation, Breusch-Pagan for heteroscedasticity) and *consequences* (biased standard errors). You should also know the basic remedies: robust standard errors, generalised least squares, and adding lagged variables.

---
## 6. Multiple Regression

So far, we have used a single independent variable (market return) to explain the dependent variable (stock return). But in practice, multiple factors affect stock returns simultaneously — market risk, size, value, momentum, and more.

**Multiple regression** extends the framework to $k$ predictors:

### Model
$$y = X\beta + \varepsilon$$

where $X$ is the $n \times (k+1)$ design matrix (including an intercept column of ones), and $\beta = [\beta_0, \beta_1, \ldots, \beta_k]^T$.

Written out for the Fama-French three-factor model:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_{i,\text{mkt}}(R_{m,t} - R_{f,t}) + \beta_{i,\text{smb}} \cdot \text{SMB}_t + \beta_{i,\text{hml}} \cdot \text{HML}_t + \varepsilon_{i,t}$$

Each coefficient measures the stock's sensitivity to a different **risk factor**, holding the other factors constant:

- $\beta_{\text{mkt}}$ = market beta (sensitivity to market risk, same as CAPM)
- $\beta_{\text{smb}}$ = size beta (sensitivity to the small-minus-big factor; positive means the stock behaves like a small-cap stock)
- $\beta_{\text{hml}}$ = value beta (sensitivity to the high-minus-low factor; positive means the stock behaves like a value stock)

> **Key Concept:** In multiple regression, each slope coefficient measures the **partial effect** of that variable — the effect of changing one X while holding all other X's constant. This is fundamentally different from simple regression, where the slope captures the *total* effect (including any indirect effects through correlated variables).

### OLS Solution (Matrix Form)

Minimizing $\|y - X\beta\|^2$ yields:

$$\hat{\beta} = (X^TX)^{-1}X^Ty$$

### Derivation

The loss function in matrix form is:

$$L(\beta) = (y - X\beta)^T(y - X\beta) = y^Ty - 2\beta^T X^T y + \beta^T X^T X \beta$$

Taking the derivative with respect to $\beta$:

$$\frac{\partial L}{\partial \beta} = -2X^Ty + 2X^TX\beta = 0$$

$$\Rightarrow X^TX\beta = X^Ty \Rightarrow \hat{\beta} = (X^TX)^{-1}X^Ty$$

### Intuition Behind the Matrix Formula

The matrix formula looks intimidating, but the intuition is the same as simple regression:

- $X^Ty$ captures the correlation between each predictor and the response (like the numerator $\sum (x_i - \bar{x})(y_i - \bar{y})$ in simple regression).
- $(X^TX)^{-1}$ accounts for the correlations *among* the predictors themselves (like the denominator $\sum (x_i - \bar{x})^2$ in simple regression, but generalised to handle multiple correlated predictors).
- The product $(X^TX)^{-1}X^Ty$ "adjusts" the raw correlations between predictors and response for the inter-correlations among predictors, giving the partial effects.

> **Common Mistake:** When predictors are highly correlated with each other (multicollinearity), $X^TX$ is nearly singular, and $(X^TX)^{-1}$ has very large entries. This causes the standard errors to blow up, making individual coefficients appear insignificant even when the model as a whole is highly significant. The solution: drop redundant predictors or use regularisation (ridge regression, LASSO).

> **CFA Exam Tip:** For the CFA exam, you do NOT need to compute the matrix formula by hand. You need to understand: (1) what the formula does conceptually, (2) how to interpret multiple regression output, and (3) the difference between $R^2$ and adjusted $R^2$ in multiple regression.

### Implementing Multiple Regression: Fama-French 3-Factor Model

The code below:

1. Implements `ols_multiple(X, y)` — the full matrix OLS solution, including standard errors, t-statistics, p-values, and the F-test.
2. Simulates 120 months (10 years) of factor returns (market, SMB, HML) and a stock whose true factor loadings are known.
3. Estimates the factor loadings and compares them to the true values.

This is a realistic scenario: an analyst at an asset management firm might run exactly this regression to understand a stock's risk profile.

> **What to watch for:** The estimated coefficients should be close to the true values. The t-statistics tell us which factor loadings are statistically significant. The $R^2$ should be higher than the simple CAPM regression because we are using three factors instead of one.

In [ ]:
def ols_multiple(X, y):
    """Multiple linear regression via OLS.
    
    Parameters
    ----------
    X : ndarray, shape (n, k) – predictor matrix (without intercept)
    y : ndarray, shape (n,)  – response
    
    Returns
    -------
    beta_hat : ndarray – coefficient estimates [intercept, b1, ..., bk]
    diagnostics : dict with R2, adj_R2, SEE, t_stats, p_values, F_stat
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n = X.shape[0]
    
    # Add intercept column
    ones = np.ones((n, 1))
    X_full = np.hstack([ones, X]) if X.ndim > 1 else np.column_stack([ones, X])
    k = X_full.shape[1] - 1  # number of predictors (excl. intercept)
    
    # OLS: beta = (X'X)^{-1} X'y
    XtX = X_full.T @ X_full
    Xty = X_full.T @ y
    beta_hat = np.linalg.solve(XtX, Xty)
    
    # Predictions and residuals
    y_hat = X_full @ beta_hat
    residuals = y - y_hat
    y_bar = np.mean(y)
    
    # Sums of squares
    SST = np.sum((y - y_bar) ** 2)
    SSE = np.sum(residuals ** 2)
    SSR = SST - SSE
    
    R2 = 1 - SSE / SST
    adj_R2 = 1 - (1 - R2) * (n - 1) / (n - k - 1)
    MSE = SSE / (n - k - 1)
    SEE = np.sqrt(MSE)
    
    # Standard errors of coefficients
    var_beta = MSE * np.linalg.inv(XtX)
    se_beta = np.sqrt(np.diag(var_beta))
    
    # T-statistics and p-values
    t_stats = beta_hat / se_beta
    df = n - k - 1
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df))
    
    # F-statistic
    MSR = SSR / k
    F_stat = MSR / MSE
    p_F = 1 - stats.f.cdf(F_stat, k, df)
    
    return beta_hat, {
        'R2': R2, 'adj_R2': adj_R2, 'SEE': SEE,
        'se_beta': se_beta, 't_stats': t_stats, 'p_values': p_values,
        'F_stat': F_stat, 'p_F': p_F,
        'y_hat': y_hat, 'residuals': residuals,
        'SST': SST, 'SSR': SSR, 'SSE': SSE,
    }


# ── Example: Fama-French style 3-factor model (simulated)
n_obs_ff = 120  # 10 years monthly

# Factor returns
market_excess = rng.normal(0.006, 0.045, n_obs_ff)
smb = rng.normal(0.002, 0.03, n_obs_ff)   # size factor
hml = rng.normal(0.003, 0.028, n_obs_ff)   # value factor

# Stock excess return with known factor loadings
true_betas_ff = np.array([0.001, 1.2, 0.5, -0.3])  # alpha, mkt, smb, hml
X_ff = np.column_stack([market_excess, smb, hml])
y_ff = true_betas_ff[0] + X_ff @ true_betas_ff[1:] + rng.normal(0, 0.02, n_obs_ff)

# Estimate
beta_hat_ff, diag_ff = ols_multiple(X_ff, y_ff)

print("Multiple Regression: 3-Factor Model")
print(f"R_stock - R_f = α + β_mkt·MKT + β_smb·SMB + β_hml·HML + ε\n")

labels = ['α (intercept)', 'β_mkt', 'β_smb', 'β_hml']
print(f"{'Coef':<15} {'True':>8} {'Estimate':>10} {'Std Err':>10} {'t-stat':>10} {'p-value':>10}")
print("-" * 65)
for i, label in enumerate(labels):
    print(f"{label:<15} {true_betas_ff[i]:>8.4f} {beta_hat_ff[i]:>10.4f} "
          f"{diag_ff['se_beta'][i]:>10.4f} {diag_ff['t_stats'][i]:>10.4f} "
          f"{diag_ff['p_values'][i]:>10.4f}")

print(f"\nR²     = {diag_ff['R2']:.4f}")
print(f"Adj R² = {diag_ff['adj_R2']:.4f}")
print(f"F-stat = {diag_ff['F_stat']:.4f}, p = {diag_ff['p_F']:.6f}")

### Interpreting the Multi-Factor Output

The regression table above is the standard format you will encounter in research reports, Bloomberg terminals, and CFA exam questions. Let's interpret each piece:

**Coefficient estimates vs true values:**
- Each estimated coefficient should be close to (but not exactly equal to) the true value. The discrepancy is sampling error.
- With 120 observations, the estimates are more precise than our earlier 60-observation simple regression.

**Statistical significance:**
- The market beta ($\beta_{\text{mkt}}$) should be highly significant — the market is the dominant risk factor for most stocks.
- The SMB loading ($\beta_{\text{smb}} = 0.5$) and HML loading ($\beta_{\text{hml}} = -0.3$) may or may not be statistically significant depending on the noise. In real-world analysis, not all factor exposures are significant.
- The intercept (alpha) is small and likely insignificant, which is consistent with the efficient market hypothesis.

**Model fit:**
- $R^2$ should be higher than the simple CAPM regression because additional relevant factors capture more variation.
- Adjusted $R^2$ penalises for the additional parameters. If the extra factors genuinely help, adjusted $R^2$ will still be higher than the CAPM $R^2$.

> **Key Concept:** The interpretation of each coefficient changes in multiple regression. $\beta_{\text{mkt}} = 1.2$ means: "holding the size and value factors constant, a 1% increase in the market excess return is associated with a 1.2% increase in the stock's excess return." The phrase "holding constant" is crucial — it is what distinguishes multiple regression from running separate simple regressions.

> **CFA Exam Tip:** If the F-test is significant but some individual t-tests are not, this suggests multicollinearity — the factors are correlated with each other, making it hard to isolate each factor's individual contribution. The model as a whole is useful, but individual factor attributions are imprecise.

### Simple vs Multiple Regression: When Does It Matter?

A natural question is: **do the extra factors in the Fama-French model actually change the market beta estimate compared to the simple CAPM regression?**

The answer depends on whether the additional factors are correlated with the market factor:

- **If the factors are uncorrelated** (orthogonal): Adding them does NOT change the market beta estimate. It only reduces the residual variance, tightening the standard errors.
- **If the factors are correlated** with the market: Adding them DOES change the market beta estimate, because the simple regression slope captured both the direct market effect and the indirect effect through the correlated omitted factors.

This is the **omitted variable bias** problem. If you omit a relevant variable that is correlated with the included variable, your slope estimate is biased.

**Example:** Suppose small stocks tend to have higher market betas (they are riskier). If you run a simple CAPM regression for a small stock, the estimated beta will be "too high" because it captures both the genuine market sensitivity AND the size effect. Adding the SMB factor separates these two effects, giving a more accurate estimate of pure market sensitivity.

> **Key Concept:** The choice between simple and multiple regression is not just about fit ($R^2$). It is about getting **unbiased estimates** of the coefficients. Omitting a relevant, correlated variable leads to biased slopes — a far more serious problem than low $R^2$.

> **Common Mistake:** Adding irrelevant variables does not cause bias (the coefficient on the irrelevant variable will just be close to zero), but it does reduce the **efficiency** of the estimates — the standard errors get slightly larger. So do not throw in every variable you can think of; include only those with a clear economic rationale.

---
## 7. Financial Application: Estimating CAPM Beta

We now arrive at the classic financial application of regression: estimating **CAPM beta**.

### The Capital Asset Pricing Model (CAPM)

The CAPM predicts that the expected excess return of any asset is proportional to its market beta:

$$E[R_i] - R_f = \beta_i \cdot (E[R_m] - R_f)$$

To estimate $\beta_i$, we run the time-series regression:

$$R_{i,t} - R_{f,t} = \alpha_i + \beta_i (R_{m,t} - R_{f,t}) + \varepsilon_{i,t}$$

The slope of this regression IS the CAPM beta.

### What Beta Tells You

Beta measures a stock's **sensitivity to systematic (market) risk**:

| Beta | Type | Interpretation | Example |
|:-----|:-----|:--------------|:--------|
| $\beta > 1$ | Aggressive | Amplifies market moves. A beta of 1.5 means the stock moves 50% *more* than the market. | Tech stocks, high-growth companies |
| $\beta = 1$ | Neutral | Moves in lockstep with the market. | Diversified large-cap fund |
| $0 < \beta < 1$ | Defensive | Dampens market moves. A beta of 0.5 means the stock moves 50% *less* than the market. | Utilities, consumer staples |
| $\beta = 0$ | Market-neutral | No systematic relationship with the market. | T-bills (by definition) |
| $\beta < 0$ | Inverse | Moves opposite to the market. Extremely rare for individual stocks. | Some gold miners, inverse ETFs |

> **Key Concept:** Beta is a regression slope, so it has all the properties we discussed: it equals $\text{Cov}(R_i, R_m) / \text{Var}(R_m)$, it has a standard error, we can test whether it is significantly different from zero (or from 1), and it comes with an $R^2$ that tells us how much of the stock's variation is explained by market risk.

### Economic Interpretation of Beta = 1.5

Suppose you estimate $\hat{\beta} = 1.5$ for a tech stock. Here is what it means concretely:

- **When the market goes up 10%, this stock is expected to go up 15%.** The extra 5% comes from the stock's higher sensitivity to the same economic forces driving the market.
- **When the market goes down 10%, this stock is expected to go down 15%.** Beta is symmetric — high beta amplifies both gains and losses.
- **The stock carries 1.5x the systematic risk of the market.** Under CAPM, investors should demand 1.5x the market risk premium as compensation.
- **In a portfolio context:** Adding this stock increases the portfolio's overall market exposure. If you want to reduce market risk, pair it with defensive (low-beta) stocks.

> **Common Mistake:** Beta measures *systematic* risk only — the risk that comes from broad market movements. It does NOT measure total risk (which includes idiosyncratic, company-specific risk). A biotech stock might have a moderate beta (0.8) but enormous total volatility due to clinical trial outcomes. Beta is relevant for diversified portfolios, where idiosyncratic risk is diversified away.

### Jensen's Alpha

The intercept $\alpha_i$ of the CAPM regression is called **Jensen's alpha**:

- $\alpha > 0$: The stock earns more than CAPM predicts — positive abnormal return. The manager (or stock) is "beating the market" on a risk-adjusted basis.
- $\alpha = 0$: The stock earns exactly what CAPM predicts — no abnormal return. This is what the efficient market hypothesis suggests.
- $\alpha < 0$: The stock earns less than CAPM predicts — negative abnormal return. The manager is underperforming.

> **CFA Exam Tip:** Alpha is the intercept, NOT the slope. This is a common confusion. $\beta$ measures risk; $\alpha$ measures *abnormal return after adjusting for risk*. A positive alpha does NOT mean the stock has high returns — it means it has high returns *relative to its risk level*.

### The Simulation Below

We simulate four stocks with different risk profiles and estimate their CAPM betas:

1. **Tech Growth** ($\beta = 1.5$): Aggressive, amplifies market moves
2. **Blue Chip** ($\beta = 1.0$): Moves with the market
3. **Utility** ($\beta = 0.5$): Defensive, dampens market moves
4. **Gold Miner** ($\beta = -0.2$): Inverse relationship with market

For each stock, we estimate $\hat{\alpha}$, $\hat{\beta}$, and $R^2$, then visualise the scatter plot with the regression line.

In [ ]:
# ── Simulate CAPM for multiple stocks
n_months = 60
rf = 0.003  # monthly risk-free rate

# Market excess returns
mkt_excess = rng.normal(0.007, 0.045, n_months)

stocks = {
    'Tech Growth':   {'alpha': 0.002, 'beta': 1.5, 'sigma': 0.04},
    'Blue Chip':     {'alpha': 0.000, 'beta': 1.0, 'sigma': 0.02},
    'Utility':       {'alpha': -0.001, 'beta': 0.5, 'sigma': 0.015},
    'Gold Miner':    {'alpha': 0.001, 'beta': -0.2, 'sigma': 0.06},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

print(f"{'Stock':<15} {'True β':>8} {'Est β':>8} {'SE(β)':>8} {'True α':>8} {'Est α':>8} {'R²':>8}")
print("-" * 67)

for idx, (name, params) in enumerate(stocks.items()):
    # Generate stock excess returns
    stock_excess = (params['alpha'] + params['beta'] * mkt_excess + 
                    rng.normal(0, params['sigma'], n_months))
    
    # Estimate CAPM
    a, b = ols_simple(mkt_excess, stock_excess)
    d = regression_diagnostics(mkt_excess, stock_excess, a, b)
    t = coefficient_tests(mkt_excess, stock_excess, a, b, d)
    
    print(f"{name:<15} {params['beta']:>8.2f} {b:>8.4f} {t['se_beta']:>8.4f} "
          f"{params['alpha']:>8.3f} {a:>8.4f} {d['R2']:>8.4f}")
    
    # Plot
    ax = axes[idx]
    ax.scatter(mkt_excess * 100, stock_excess * 100, color=PRIMARY, alpha=0.5, 
               edgecolor='white', s=40)
    x_line = np.linspace(mkt_excess.min(), mkt_excess.max(), 100)
    ax.plot(x_line * 100, (a + b * x_line) * 100, color=SECONDARY, linewidth=2)
    ax.set_title(f'{name}: β̂ = {b:.2f} (true = {params["beta"]:.1f}), R² = {d["R2"]:.2f}')
    ax.set_xlabel('Market Excess Return (%)')
    ax.set_ylabel('Stock Excess Return (%)')

plt.tight_layout()
plt.show()

### Interpreting the Four CAPM Regressions

The table and scatter plots above reveal several important patterns:

**Tech Growth ($\beta = 1.5$):**
- The steep upward slope confirms this is an aggressive stock.
- The scatter is moderately tight — the market explains a reasonable fraction of the stock's return variation.
- With $\sigma_{\varepsilon} = 4\%$, there is substantial idiosyncratic risk, so $R^2$ is not extremely high.

**Blue Chip ($\beta = 1.0$):**
- The slope is approximately 1:1 — the stock moves roughly in line with the market.
- Lower idiosyncratic volatility ($\sigma_{\varepsilon} = 2\%$) means a tighter scatter and higher $R^2$.

**Utility ($\beta = 0.5$):**
- The gentle slope confirms this is a defensive stock.
- Even though the true beta is far from zero, the $R^2$ may be moderate because the idiosyncratic noise ($\sigma_{\varepsilon} = 1.5\%$) is comparable in magnitude to the systematic component ($0.5 \times 4.5\% = 2.25\%$).

**Gold Miner ($\beta = -0.2$):**
- The slightly downward slope shows the inverse relationship with the market.
- The $R^2$ is likely very low — the market explains almost none of the variation in this stock's returns. Most of the action comes from gold-specific factors (which are in the error term).
- The large idiosyncratic volatility ($\sigma_{\varepsilon} = 6\%$) means the scatter is very wide.

> **Key Concept:** $R^2$ varies dramatically across different types of stocks. A large-cap stock that closely tracks the index might have $R^2 > 0.5$, while a gold miner or a speculative biotech might have $R^2 < 0.05$. Low $R^2$ does NOT mean the regression is useless — the beta estimate can still be informative and statistically significant.

> **CFA Exam Tip:** Beta is estimated from historical data, but it is used to make forward-looking predictions about expected return. The assumption is that systematic risk exposure is relatively stable over time — which is a strong assumption! In practice, betas change: a growth company may have $\beta > 1.5$ when young and $\beta < 1.0$ as it matures. Practitioners often use "adjusted beta" formulas (like Blume's adjustment: $\beta_{\text{adj}} = 0.33 + 0.67 \times \beta_{\text{raw}}$) to shrink extreme betas toward 1.0.

### The Security Market Line

The **Security Market Line (SML)** is the graphical representation of CAPM. It plots **expected return** (Y-axis) against **beta** (X-axis). According to CAPM:

$$E[R_i] = R_f + \beta_i \cdot (E[R_m] - R_f)$$

This is a straight line with:
- **Intercept** = $R_f$ (the risk-free rate)
- **Slope** = $E[R_m] - R_f$ (the market risk premium)

**Stocks that plot above the SML** have positive alpha — they offer more return than their risk level warrants. They are underpriced (a buying opportunity, if the alpha is genuine).

**Stocks that plot below the SML** have negative alpha — they offer less return than their risk level warrants. They are overpriced.

**Stocks that plot on the SML** are fairly priced according to CAPM.

> **Key Concept:** The SML is a line in **expected return vs beta** space, NOT in **expected return vs total risk** space. That distinction matters: the SML uses beta (systematic risk only), while the Capital Market Line (CML) uses standard deviation (total risk). The SML applies to all assets; the CML applies only to efficient portfolios.

The plot below shows the SML and our four simulated stocks. Their vertical distance from the SML is their Jensen's alpha.

In [ ]:
# ── Security Market Line (SML) visualization
fig, ax = plt.subplots(figsize=(10, 6))

# SML: E[R] = Rf + beta * (E[Rm] - Rf)
market_premium = 0.06 / 12  # monthly
beta_range = np.linspace(-0.5, 2.0, 100)
sml = rf + beta_range * market_premium

ax.plot(beta_range, sml * 1200, color=PRIMARY, linewidth=2, label='SML: E[R] = Rf + β·MRP')

# Plot each stock
colors_stocks = [SECONDARY, TERTIARY, ACCENT, 'purple']
for (name, params), color in zip(stocks.items(), colors_stocks):
    expected_ret = rf + params['beta'] * market_premium + params['alpha']
    ax.plot(params['beta'], expected_ret * 1200, 'o', color=color, markersize=12,
            label=f"{name} (β={params['beta']:.1f})")

ax.set_xlabel('Beta')
ax.set_ylabel('Expected Return (% annualized)')
ax.set_title('Security Market Line (CAPM)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Reading the Security Market Line Plot

The blue line is the theoretical SML — the relationship CAPM predicts between beta and expected return. Each dot is one of our four simulated stocks.

- **Tech Growth** (coral, $\beta = 1.5$): Plots slightly above the SML because we gave it a positive alpha ($\alpha = 0.002$). It offers more return than its risk level alone would justify.
- **Blue Chip** (green, $\beta = 1.0$): Plots very close to the SML ($\alpha = 0$). This stock is fairly priced per CAPM.
- **Utility** (gold, $\beta = 0.5$): Plots slightly below the SML ($\alpha = -0.001$). It underperforms on a risk-adjusted basis.
- **Gold Miner** (purple, $\beta = -0.2$): Plots above the SML ($\alpha = 0.001$). Despite negative beta, it offers a small positive alpha.

**The fundamental insight:** In an efficient market, all stocks should plot ON the SML (alpha = 0 for all). Active managers seek stocks that plot ABOVE the SML — stocks where the alpha is genuinely positive, not just a statistical artifact.

> **Common Mistake:** Do not confuse "high return" with "positive alpha." A high-beta stock can have high returns and still have negative alpha (it is below the SML). Conversely, a low-beta stock can have modest returns and still have positive alpha (it is above the SML). Alpha is always *relative to the risk taken*.

> **CFA Exam Tip:** If given a stock's beta and its actual return, you can calculate Jensen's alpha as: $\alpha = R_i - [R_f + \beta_i(R_m - R_f)]$. This is simply: actual return minus CAPM-predicted return.

### Practical Considerations for Beta Estimation

In the real world, several choices affect your beta estimate:

**1. Choice of time period:**
- Using 5 years of monthly data (60 observations) is the industry standard (Bloomberg default).
- Shorter periods capture more recent risk profile but have more estimation error.
- Longer periods provide more precision but may include regime changes (e.g., a tech company that pivoted its business model).

**2. Choice of frequency:**
- Monthly returns are most common for beta estimation.
- Daily returns give more observations but introduce microstructure noise (bid-ask bounce, non-synchronous trading).
- Weekly returns are a compromise, reducing microstructure effects while providing more data than monthly.

**3. Choice of market index:**
- S&P 500 is the most common choice for US stocks.
- For international stocks, use a local or global index.
- The choice matters: a stock may have $\beta = 1.2$ relative to the S&P 500 but $\beta = 0.9$ relative to the MSCI World.

**4. Beta stationarity:**
- Betas are NOT constant over time. A company's risk profile changes as it grows, takes on debt, enters new markets, or faces regulatory changes.
- Rolling-window estimation (e.g., re-estimate beta every month using the most recent 60 months) captures time variation.
- Blume's adjustment ($\beta_{\text{adj}} = 0.33 + 0.67 \times \beta_{\text{raw}}$) shrinks extreme betas toward 1.0, reflecting the empirical tendency of betas to regress toward the market average over time.

> **CFA Exam Tip:** Be aware that beta estimation involves judgement calls (time period, frequency, index). Two analysts estimating beta for the same stock can get different answers. The exam may ask you about the pros and cons of different estimation choices.

**5. Statistical significance is not the same as economic significance:**
- With enough data, even a tiny beta (say, 0.02) can be "statistically significant" — meaning we can reject $H_0: \beta = 0$.
- But a beta of 0.02 is economically meaningless for portfolio construction.
- Always consider both statistical significance (is the estimate reliably different from zero?) and economic significance (is the estimate large enough to matter?).

> **Common Mistake:** Reporting that a coefficient is "significant" without specifying the significance level. Always state whether you are using 1%, 5%, or 10%. And always report the actual p-value when possible — it is more informative than a simple pass/fail at an arbitrary threshold.

---
## Putting It All Together: A Regression Analysis Checklist

When you encounter a regression problem — whether on the CFA exam or in professional practice — follow this systematic workflow:

### Step 1: Specify the Model
- What is the dependent variable? What are the independent variables?
- Is there a clear economic theory motivating the relationship? (In finance, always start with theory — CAPM, Fama-French, APT, etc.)
- Should you use simple or multiple regression?

### Step 2: Examine the Data
- Plot a scatter diagram (for simple regression) or a correlation matrix (for multiple regression).
- Look for outliers, nonlinearity, and clusters.
- Check the sample size: is it large enough for reliable inference? (As a rough guide, you want at least 30 observations for simple regression, and at least 10-20 observations per predictor for multiple regression.)

### Step 3: Estimate the Coefficients
- Run OLS regression.
- Record the coefficient estimates, standard errors, t-statistics, and p-values.

### Step 4: Evaluate the Model
- Check $R^2$ and adjusted $R^2$: how much variation is explained?
- Check the F-statistic: is the overall regression significant?
- Check individual t-statistics: which coefficients are significant?

### Step 5: Check the Assumptions
- Plot residuals vs fitted values (look for patterns, fan shapes).
- Check the Q-Q plot (look for departures from normality).
- Compute the Durbin-Watson statistic (test for autocorrelation).
- Run the Breusch-Pagan test (test for heteroscedasticity).

### Step 6: Interpret Economically
- Translate the numbers into financial language.
- Does the sign of each coefficient make economic sense?
- Is the magnitude reasonable? (A beta of 15 for a stock is almost certainly an error.)
- Are the results robust? (Would they change with a different time period, frequency, or specification?)

> **Key Concept:** The best regression analysts are not the ones who can compute the fastest — they are the ones who can **interpret** the results most thoughtfully. Numbers without economic context are meaningless. Always ask: "What does this tell me about the financial world?"

### Common Exam Question Types

The CFA exam tests regression in several predictable ways:

1. **Interpretation questions:** "Given the following regression output, interpret the slope coefficient." Always state the units and direction.
2. **Hypothesis testing:** "Is the slope significantly different from zero at the 5% level?" Compare the t-statistic to the critical value, or check if the p-value is below 0.05.
3. **ANOVA calculations:** "Given SSR = 120, SSE = 80, n = 50, k = 2, compute the F-statistic." Use SST = SSR + SSE, then MSR = SSR/k, MSE = SSE/(n-k-1), F = MSR/MSE.
4. **Assumption violations:** "The residual plot shows a fan shape. Which assumption is violated?" Heteroscedasticity.
5. **Prediction:** "Using the regression equation, predict Y when X = 5." Plug into $\hat{Y} = \hat{\beta}_0 + \hat{\beta}_1 \times 5$.
6. **CAPM beta:** "A stock's beta is 1.3 and the market risk premium is 6%. What is the expected excess return?" Answer: $1.3 \times 6\% = 7.8\%$.

> **CFA Exam Tip:** Time management matters. Regression questions often have multiple sub-parts. Read all parts before starting — sometimes a later part gives you information needed for an earlier part. Also, always check whether the question asks for a one-tailed or two-tailed test.

---
## Summary: What You Should Take Away

Linear regression is the foundation of quantitative finance. Here is what we covered:

1. **Regression finds the best-fitting line** through data by minimising squared vertical distances (OLS).
2. **The slope** $\hat{\beta} = \text{Cov}(X,Y) / \text{Var}(X)$ — in CAPM, this is the stock's beta.
3. **The intercept** $\hat{\alpha} = \bar{Y} - \hat{\beta}\bar{X}$ — in CAPM, this is Jensen's alpha.
4. **$R^2$** measures the percentage of variation explained, ranging from 0 (useless) to 1 (perfect).
5. **Hypothesis tests** (t-tests, F-tests) tell us whether the relationship is statistically significant.
6. **The ANOVA table** decomposes total variation into explained (SSR) and unexplained (SSE) parts.
7. **Four assumptions** (linearity, independence, homoscedasticity, normality) must be checked for valid inference.
8. **Multiple regression** extends to many predictors using the matrix formula $\hat{\beta} = (X^TX)^{-1}X^Ty$.
9. **CAPM beta** is literally a regression slope, and the entire CAPM framework rests on regression analysis.

> **Key Concept:** Regression is not just a statistical technique — it is a way of thinking about relationships in data. Every time you see a financial model, ask: "What is the regression being run here? What is the dependent variable? What are the independent variables? What assumptions are being made?"

---
## References

The following texts provide deeper coverage of the topics in this notebook:

1. **CFA Institute**, *CFA Program Curriculum Level I*, Quantitative Methods: Correlation and Regression.
2. **DeFusco, R., McLeavey, D., Pinto, J., & Runkle, D.**, *Quantitative Investment Analysis*, 3rd ed., CFA Institute/Wiley, 2015.
3. **Greene, W.**, *Econometric Analysis*, 8th ed., Pearson, 2018.
4. **Wooldridge, J.**, *Introductory Econometrics: A Modern Approach*, 7th ed., Cengage, 2020.
5. **Fama, E. & French, K.**, "Common Risk Factors in the Returns on Stocks and Bonds", *Journal of Financial Economics*, 33(1), 1993.
6. **Sharpe, W.**, "Capital Asset Prices: A Theory of Market Equilibrium Under Conditions of Risk", *Journal of Finance*, 19(3), 1964.